# Sesión 2 — Entrenar y exportarEste notebook entrena un modelo y **exporta el artefacto de despliegue**.Lo importante de esta sesión no es el modelo: es lo que sale de aquí. Corre las celdas deentrenamiento sin detenerte demasiado; donde vamos a pasar el rato es en la exportación.```artifacts/├── pipeline.joblib     el Pipeline COMPLETO: preprocesamiento + estimador│                       + transformación del target├── metadata.json       el CONTRATO, en texto legible└── example.json        un input válido y su predicción de referencia```

## 1. Lo que necesitamos

In [ ]:
import hashlib
import json
import pathlib
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Las diez featuresNo son las diez que dan el mejor RMSE. Son **las diez que un vendedor puede contestar sinmedir la casa**. Esa diferencia es una decisión de producto, y es la primera de esta sesión.

In [ ]:
RAIZ = pathlib.Path(__file__).resolve().parent.parent
DATOS = RAIZ / "data" / "train.csv"
ARTEFACTOS = RAIZ / "artifacts"

MODEL_VERSION = "1.0.0"
SEMILLA = 42

# Las diez features que un vendedor real conoce. No son las diez que dan el
# mejor RMSE: son las diez que alguien puede contestar sin medir la casa.
NUMERICAS = [
    "GrLivArea",
    "OverallQual",
    "YearBuilt",
    "TotalBsmtSF",
    "GarageCars",
    "FullBath",
    "BedroomAbvGr",
    "LotArea",
]
CATEGORICAS = ["Neighborhood", "KitchenQual"]
FEATURES = NUMERICAS + CATEGORICAS
TARGET = "SalePrice"

## 3. El pipeline completo — la pieza central del móduloMira dónde vive cada cosa:- El **preprocesamiento** está dentro del pipeline, no en celdas sueltas de este notebook.- La **transformación del target** también, envuelta en `TransformedTargetRegressor`.Por eso el servicio va a poder pasarle un `DataFrame` crudo y recibir pesos, sin saber nadade imputaciones, codificación ni logaritmos.La alternativa —aplicar `np.expm1()` en el código de Flask— pone conocimiento del modelo enquien lo consume. El siguiente que use este artefacto lo va a olvidar y va a servir preciosde 12.3 dólares.

In [ ]:
def construir_pipeline():
    """El pipeline completo, en un solo objeto.

    Esta es la pieza central del modulo. El preprocesamiento NO queda fuera en
    celdas sueltas del notebook: viaja dentro del artefacto, asi que el
    servicio puede pasarle un DataFrame crudo y no necesita saber nada de
    imputaciones ni de codificacion.
    """
    preproceso = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imputar", SimpleImputer(strategy="median")),
                        ("escalar", StandardScaler()),
                    ]
                ),
                NUMERICAS,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputar", SimpleImputer(strategy="most_frequent")),
                        (
                            "codificar",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                CATEGORICAS,
            ),
        ]
    )

    modelo = Pipeline(
        [
            ("preproceso", preproceso),
            (
                "estimador",
                RandomForestRegressor(n_estimators=300, random_state=SEMILLA, n_jobs=-1),
            ),
        ]
    )

    # La transformacion del target vive DENTRO del artefacto.
    #
    # La alternativa seria aplicar np.expm1() en el codigo de Flask, y es peor:
    # pone conocimiento del modelo en quien lo consume. El siguiente que use
    # este artefacto lo va a olvidar y va a servir precios de 12.3 dolares.
    #
    # Asi, predict() devuelve pesos. Punto.
    return TransformedTargetRegressor(
        regressor=modelo, func=np.log1p, inverse_func=np.expm1
    )

## 4. Tres conjuntos, no dosEntrenamiento para aprender, validación para decidir, prueba para reportar. **La prueba setoca una sola vez**, al final.

In [ ]:
ARTEFACTOS.mkdir(exist_ok=True)
df = pd.read_csv(DATOS)
X = df[FEATURES]
y = df[TARGET]

# Tres conjuntos, no dos: entrenamiento para aprender, validacion para
# decidir, prueba para reportar. La prueba se toca UNA vez.
X_ent, X_resto, y_ent, y_resto = train_test_split(
    X, y, test_size=0.3, random_state=SEMILLA
)
X_val, X_prueba, y_val, y_prueba = train_test_split(
    X_resto, y_resto, test_size=0.5, random_state=SEMILLA
)

## 5. Entrenar

In [ ]:
pipeline = construir_pipeline()
pipeline.fit(X_ent, y_ent)

## 6. Medir

In [ ]:
def metricas(X_, y_):
    pred = pipeline.predict(X_)
    return {
        "rmse": round(float(np.sqrt(mean_squared_error(y_, pred))), 1),
        "mae": round(float(mean_absolute_error(y_, pred)), 1),
        "r2": round(float(r2_score(y_, pred)), 4),
    }

m_val = metricas(X_val, y_val)
m_prueba = metricas(X_prueba, y_prueba)

In [ ]:
print('validacion:', m_val)
print('prueba    :', m_prueba)

## 7. Comprobación antes de exportar`predict()` **tiene que devolver pesos**, no logaritmos. Si aquí sale un número entre 10 y 14,la transformación del target quedó fuera del artefacto y el servicio va a servir basura.

In [ ]:
prediccion = pipeline.predict(X_prueba.iloc[[0]])[0]
print(f"prediccion: {prediccion:,.2f}")
assert 10_000 < prediccion < 1_000_000, "esto no son pesos: revisa el TransformedTargetRegressor"
print("OK: predict() devuelve unidades del dominio")

# ─────────────────────────────────────────────# Aquí empieza lo que de verdad importa# ─────────────────────────────────────────────## 8. Exportar el pipelineUn solo archivo, con todo dentro.

In [ ]:
# --- pipeline.joblib ---
ruta_pipeline = ARTEFACTOS / "pipeline.joblib"
joblib.dump(pipeline, ruta_pipeline)
hash_artefacto = hashlib.sha256(ruta_pipeline.read_bytes()).hexdigest()[:12]

## 9. Las importancias, traducidas a las features originalesEl modelo ve las columnas **después** del one-hot: `Neighborhood_NAmes`,`Neighborhood_CollgCr`... Para el contrato queremos la importancia de `Neighborhood`completo, así que sumamos las columnas que salieron de él.Esto se hace **aquí y no en el servicio**: el servicio no debería tener que hurgar dentro deun pipeline anidado para saber qué feature pesa más.

In [ ]:
def importancias_por_feature(ttr):
    """Traduce las importancias del modelo a las features originales.

    El modelo ve las columnas DESPUES del one-hot: 'Neighborhood_NAmes',
    'Neighborhood_CollgCr'... Para el contrato queremos la importancia de
    'Neighborhood' completo, asi que sumamos las columnas que salieron de el.
    """
    estimador = ttr.regressor_.named_steps["estimador"]
    nombres = ttr.regressor_.named_steps["preproceso"].get_feature_names_out()
    pesos = estimador.feature_importances_

    acumulado = {f: 0.0 for f in FEATURES}
    for nombre, peso in zip(nombres, pesos):
        sin_prefijo = nombre.split("__", 1)[1]
        for f in FEATURES:
            if sin_prefijo == f or sin_prefijo.startswith(f + "_"):
                acumulado[f] += float(peso)
                break
    return dict(sorted(acumulado.items(), key=lambda kv: kv[1], reverse=True))

## 10. `metadata.json` — el contratoUn archivo, tres consumidores:```                    ┌──▶ el servicio VALIDA la entrada contra él   metadata.json ───┼──▶ la Model Card se RENDERIZA de él                    └──▶ la rúbrica de tu reto se EVIDENCIA con él```Fíjate en `sklearn_version`. `joblib` no es un formato estable: un artefacto exportado conuna versión y cargado con otra puede fallar, o —peor— cargar y devolver números distintos sinavisar. El servicio va a comparar esa versión al arrancar.

In [ ]:
# --- metadata.json: el contrato ---
features_contrato = []
for f in NUMERICAS:
    features_contrato.append(
        {
            "name": f,
            "type": "num",
            "min": float(df[f].min()),
            "max": float(df[f].max()),
            "median": float(df[f].median()),
        }
    )
for f in CATEGORICAS:
    features_contrato.append(
        {
            "name": f,
            "type": "cat",
            "allowed": sorted(df[f].dropna().unique().tolist()),
        }
    )

metadata = {
    "model_version": MODEL_VERSION,
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "artifact_hash": hash_artefacto,
    "algorithm": "RandomForestRegressor(n_estimators=300)",
    "target": TARGET,
    "target_transform": "log1p",
    "features": features_contrato,
    "splits": {
        "train": int(len(X_ent)),
        "validation": int(len(X_val)),
        "test": int(len(X_prueba)),
    },
    "metrics": {"validation": m_val, "test": m_prueba},
    "feature_importances": {
        k: round(v, 4) for k, v in importancias_por_feature(pipeline).items()
    },
    # Campos que este modulo deja vacios y que cada equipo llena en su reto.
    # Las vistas del tablero ya los saben leer: aparecen como "sin datos".
    "model_comparison": [],
    "hyperparameter_experiments": [],
}
(ARTEFACTOS / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + "\n"
)

## 11. `example.json` — el smoke testUn input válido y la predicción que este modelo le da **ahora mismo**. Es lo que va apermitir comprobar que el servicio devuelve el mismo número que este notebook.Ese es `tests/test_paridad.py`, y es el único test que de verdad importa en este módulo.

In [ ]:
# --- example.json: el smoke test ---
fila = X_prueba.iloc[0]
ejemplo = {
    "input": {
        k: (int(v) if isinstance(v, (np.integer,)) else
            float(v) if isinstance(v, (np.floating,)) else v)
        for k, v in fila.to_dict().items()
    },
    "prediction": round(float(pipeline.predict(X_prueba.iloc[[0]])[0]), 2),
    "model_version": MODEL_VERSION,
}
(ARTEFACTOS / "example.json").write_text(
    json.dumps(ejemplo, indent=2, ensure_ascii=False) + "\n"
)

## 12. Resumen

In [ ]:
print("Artefacto exportado en artifacts/")
print(f"  splits          : {metadata['splits']}")
print(f"  RMSE validacion : {m_val['rmse']:,.0f}")
print(f"  RMSE prueba     : {m_prueba['rmse']:,.0f}")
print(f"  R2 prueba       : {m_prueba['r2']}")
print(f"  sklearn         : {metadata['sklearn_version']}")
print(f"  hash            : {hash_artefacto}")
print(f"  prediccion ref  : {ejemplo['prediction']:,.2f}")
print()
print("  importancias:")
for k, v in list(metadata["feature_importances"].items())[:5]:
    print(f"    {k:<16} {v}")

## 13. Qué NO fue al artefactoVale la pena decirlo en voz alta:- **Los datos de entrenamiento.** El artefacto lleva el modelo, no el CSV.- **Credenciales.** Nunca, en ningún artefacto.- **Rutas absolutas de esta máquina.** Es el error clásico: un `Pipeline` que congela  `/Users/tu-nombre/...` dentro del pickle y falla en el servidor. `test_paridad.py` lo  comprueba.Ahora ve a `docs/s2-guia.md` y sigue con el servicio.